# Aim of this code is to use np.stack (simple stacking) for aligning 222 brain.

In [1]:
# STEPS:
# 1) Read the images
# 2) Resize the image (if required)
# 3) Sort the jpg images based on the file name
# 4) Extract the blue channel
# 5) Align the centroid
# 6) np.stack
# 7) save the image in .nrrd file

In [2]:
import os
import re
import cv2
import numpy as np
import nrrd
from tqdm import tqdm

In [3]:
# Define the input folder containing JPG images
input_folder = '/home/projects/registration/'


In [4]:
# Set the final desired size to 1000x1000 after downsampling
final_height = 1000
final_width = 1000

In [5]:
# Function to extract the image number from the filename
def extract_image_number(filename):
    pattern = re.compile(r'_(\d+)_original\.jpg$')
    match = pattern.search(filename)
    if match:
        return int(match.group(1))
    else:
        print(f"Warning: No number found in filename {filename}. Skipping.")
        return None

In [6]:
# List and sort all JPG images in the folder based on the extracted image number
image_files = [f for f in os.listdir(input_folder) if f.endswith(".jpg")]
image_files.sort(key=lambda f: extract_image_number(f) or float('inf'))

In [7]:
# Function to read, resize (downsample by 3), and extract the blue channel from an image
def extract_blue_channel(img_path, final_width, final_height):
    image = cv2.imread(img_path)
    if image is None:
        raise ValueError(f"Unable to read image {img_path}.")
    
    # Calculate the downsampled size (3x reduction)
    downsampled_height = final_height * 3
    downsampled_width = final_width * 3
    
    # Downsample the image first by resizing to the larger dimensions
    resized_image = cv2.resize(image, (downsampled_width, downsampled_height))
    
    # Then resize to the final desired dimensions (1000x1000)
    resized_image = cv2.resize(resized_image, (final_width, final_height))
    
    # Extract the blue channel (OpenCV loads images in BGR format)
    blue_channel = resized_image[:, :, 0]
    
    # Ensure the blue channel is of dtype uint8
    blue_channel = blue_channel.astype(np.uint8)
    
    return blue_channel

In [8]:
# Function to align centroids of images
def align_centroid(image, reference_image):
    # Compute the centroid of the current image
    moments = cv2.moments(image)
    if moments["m00"] != 0:
        cx = int(moments["m10"] / moments["m00"])
        cy = int(moments["m01"] / moments["m00"])
    else:
        cx, cy = image.shape[1] // 2, image.shape[0] // 2  # Fallback to center
    
    # Compute the centroid of the reference image
    ref_moments = cv2.moments(reference_image)
    if ref_moments["m00"] != 0:
        ref_cx = int(ref_moments["m10"] / ref_moments["m00"])
        ref_cy = int(ref_moments["m01"] / ref_moments["m00"])
    else:
        ref_cx, ref_cy = reference_image.shape[1] // 2, reference_image.shape[0] // 2  # Fallback to center
    
    # Calculate the translation required to align centroids
    dx = ref_cx - cx
    dy = ref_cy - cy
    
    # Translate the image to align centroids
    translation_matrix = np.float32([[1, 0, dx], [0, 1, dy]])
    aligned_image = cv2.warpAffine(image, translation_matrix, (image.shape[1], image.shape[0]))
    
    return aligned_image

In [9]:
# Extract the blue channel of the first image (reference image for centroid alignment)
reference_image_path = os.path.join(input_folder, image_files[0])
reference_blue_channel = extract_blue_channel(reference_image_path, final_width, final_height)

# Initialize the list for aligned blue channels
aligned_blue_channels = [reference_blue_channel]  # Start with the reference image

In [10]:
# Align all images based on their centroid relative to the reference image
for filename in tqdm(image_files[1:], desc="Processing images", unit="image"):
    img_path = os.path.join(input_folder, filename)
    
    # Extract the blue channel
    blue_channel = extract_blue_channel(img_path, final_width, final_height)
    
    # Align the current image to the reference image based on centroid
    aligned_image = align_centroid(blue_channel, reference_blue_channel)
    
    # Append the aligned image to the list
    aligned_blue_channels.append(aligned_image)

# Convert the list of aligned images to a NumPy array
aligned_blue_channels_array = np.stack(aligned_blue_channels, axis=0).astype(np.uint8)


Processing images: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 17/17 [00:03<00:00,  4.99image/s]


In [11]:
# Save the final aligned stack as a .nrrd file
nrrd.write('SimpleStacking.nrrd', aligned_blue_channels_array)

print(f"Final shape of the aligned image stack: {aligned_blue_channels_array.shape}")
print(f"Data type of the aligned image stack: {aligned_blue_channels_array.dtype}")

Final shape of the aligned image stack: (18, 1000, 1000)
Data type of the aligned image stack: uint8
